**Stage 4: Dataset Creation**

For each window position `k`, the input is the raw KPI window `X[k]`; targets are the current consensus label `Y_current[k]` and next consensus label `Y_next[k+1]`. The final sample count is one less than the number of windows because the last window has no next state.

In [1]:
import json
import numpy as np
import pandas as pd

RAW_FEATURES = [
    "time", "phy_mcs", "mac_dl_cqi", "mac_dl_ri", "mac_dl_pmi",
    "mac_ul_buffer", "mac_n_prb", "rsrq", "rsrp", "rssi",
    "dl_sinr", "se", "dl_bler", "delay"
]

LABEL_HEADS = [
    "link_quality", "congestion", "latency",
    "interference", "reliability", "scheduler"
]

windowed_output = pd.read_csv("windowed_output_semantic.csv")
window_consensus_labels = pd.read_csv("window_consensus_labels.csv")


# Label Encodings used in this dataset creation
# ML model can predict encoded features easily than semantic labelling approach
LABEL_ENCODINGS = {
    "link_quality": {"Excellent": 0, "Good": 1, "Fair": 2, "Poor": 3},
    "congestion": {"Underutilized": 0, "Moderate": 1, "Busy": 2, "Congested": 3},
    "latency": {"Unknown": 0, "Realtime": 1, "Normal": 2, "Risk": 3, "Critical": 4},
    "interference": {"Low": 0, "Moderate": 1, "Risk": 2},
    "reliability": {"Reliable": 0, "Warning": 1, "Critical": 2},
    "scheduler": {"Excellent": 0, "Moderate": 1, "Poor": 2},
}

dataset_rows = []
encoded_rows = []
for idx in range(len(window_consensus_labels) - 1):
    flat_input = {}
    for position in range(5):
        for feature in RAW_FEATURES:
            flat_input[f"x_{feature}_{position}"] = windowed_output.loc[idx, f"{feature}_{position}"]

    current = {f"y_current_{head}": window_consensus_labels.loc[idx, head] for head in LABEL_HEADS}
    next_ = {f"y_next_{head}": window_consensus_labels.loc[idx + 1, head] for head in LABEL_HEADS}
    dataset_rows.append({**flat_input, **current, **next_})

    encoded_current = {
        f"y_current_{head}": LABEL_ENCODINGS[head][window_consensus_labels.loc[idx, head]]
        for head in LABEL_HEADS
    }
    encoded_next = {
        f"y_next_{head}": LABEL_ENCODINGS[head][window_consensus_labels.loc[idx + 1, head]]
        for head in LABEL_HEADS
    }
    encoded_rows.append({**flat_input, **encoded_current, **encoded_next})

stage4_dataset = pd.DataFrame(dataset_rows)
stage4_dataset_encoded = pd.DataFrame(encoded_rows)

x_cols = [f"x_{feature}_{position}" for position in range(5) for feature in RAW_FEATURES]
y_current_cols = [f"y_current_{head}" for head in LABEL_HEADS]
y_next_cols = [f"y_next_{head}" for head in LABEL_HEADS]

X = stage4_dataset[x_cols].to_numpy(dtype=np.float32).reshape(len(stage4_dataset), 5, len(RAW_FEATURES))
Y_current = stage4_dataset_encoded[y_current_cols].to_numpy(dtype=np.int64)
Y_next = stage4_dataset_encoded[y_next_cols].to_numpy(dtype=np.int64)

stage4_dataset.to_csv("dataset.csv", index=False)
stage4_dataset_encoded.to_csv("dataset_encoded.csv", index=False)
np.savez("semantic_state_dataset.npz", X=X, Y_current=Y_current, Y_next=Y_next)

with open("label_encodings.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_ENCODINGS, f, indent=2)

print("stage4_dataset:", stage4_dataset.shape)
print("X:", X.shape)
print("Y_current:", Y_current.shape)
print("Y_next:", Y_next.shape)
display(stage4_dataset.head())


stage4_dataset: (5994, 82)
X: (5994, 5, 14)
Y_current: (5994, 6)
Y_next: (5994, 6)


,x_time_0,x_phy_mcs_0,x_mac_dl_cqi_0,x_mac_dl_ri_0,x_mac_dl_pmi_0,x_mac_ul_buffer_0,x_mac_n_prb_0,x_rsrq_0,x_rsrp_0,x_rssi_0,...,y_current_latency,y_current_interference,y_current_reliability,y_current_scheduler,y_next_link_quality,y_next_congestion,y_next_latency,y_next_interference,y_next_reliability,y_next_scheduler
0,1.743169e+12,0.0,9.0,1.0,3.0,4.666667,2.0,-6.700000,-88.0,-68.0,...,Unknown,Low,Reliable,Excellent,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
1,1.743169e+12,0.0,9.0,1.0,3.0,4.666667,4.0,-6.733333,-88.0,-68.0,...,Unknown,Low,Reliable,Excellent,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
2,1.743169e+12,0.0,9.0,1.0,3.0,0.000000,6.0,-6.733333,-88.0,-68.0,...,Unknown,Low,Reliable,Excellent,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
3,1.743169e+12,0.0,9.0,1.0,3.0,0.000000,12.0,-6.866667,-88.0,-68.0,...,Unknown,Low,Reliable,Excellent,Excellent,Underutilized,Unknown,Low,Reliable,Excellent
4,1.743169e+12,0.0,9.0,1.0,3.0,0.000000,16.0,-6.866667,-88.0,-68.0,...,Unknown,Low,Reliable,Excellent,Excellent,Underutilized,Realtime,Low,Reliable,Excellent
